# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [4]:
# This cell is for defining the core matrix class

class Matrix:
    def __init__(self, data, m=None):
        if isinstance(data, int) and isinstance(m, int):
            self._rows = data # zero matrix
            self._cols = m
            self._data = [[0.0 for _ in range(m)] for _ in range(data)]
        elif isinstance(data, list):
            if not data or not all(isinstance(row, list) for row in data):
                raise ValueError("Invalid matrix data.")
            row_length = len(data[0])
            if any(len(row) != row_length for row in data):
                raise ValueError("Matrix must be rectangular.")
            self._rows = len(data)
            self._cols = row_length
            self._data = [list(map(float, row)) for row in data]
        else:
            raise ValueError("Invalid initialization.")

    def shape(self):
        return (self._rows, self._cols)

    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            return self._data[i][j]
        return self._data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self._data[i][j] = float(value)
        else:
            self._data[key] = value

    def to_list(self):
        return [row[:] for row in self._data]

    def transpose(self):
        return Matrix([[self._data[i][j] for i in range(self._rows)]
                       for j in range(self._cols)])

    def row(self, n):
        return Matrix([self._data[n]])

    def column(self, n):
        return Matrix([[self._data[i][n]] for i in range(self._rows)])

    def block(self, n0, n1, m0, m1):
        return Matrix([row[n0:n1] for row in self._data[m0:m1]])

    def scalar_mult(self, c):
        return Matrix([[c * self._data[i][j]
                        for j in range(self._cols)]
                        for i in range(self._rows)])

    def add(self, other):
        if self.shape() != other.shape():
            raise ValueError("Size mismatch.")
        return Matrix([[self[i, j] + other[i, j]
                        for j in range(self._cols)]
                        for i in range(self._rows)])

    def sub(self, other):
        return self.add(other.scalar_mult(-1))

    def mat_mult(self, other):
        if self._cols != other._rows:
            raise ValueError("Incompatible sizes.")
        result = Matrix(self._rows, other._cols)
        for i in range(self._rows):
            for j in range(other._cols):
                for k in range(self._cols):
                    result[i, j] += self[i, k] * other[k, j]
        return result

    def element_mult(self, other):
        if self.shape() != other.shape():
            raise ValueError("Size mismatch.")
        return Matrix([[self[i, j] * other[i, j]
                        for j in range(self._cols)]
                        for i in range(self._rows)])

    def equals(self, other):
        if self.shape() != other.shape():
            return False
        for i in range(self._rows):
            for j in range(self._cols):
                if self[i, j] != other[i, j]:
                    return False
        return True

    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __mul__(self, other):
        if isinstance(other, Matrix):
            return self.mat_mult(other)
        return self.scalar_mult(other)

    def __rmul__(self, other):
        return self.scalar_mult(other)

    def __eq__(self, other):
        return self.equals(other)

    def __repr__(self):
        return "\n".join(str(row) for row in self._data)
    


In [5]:
# This cell is for defining special functions

def constant(n, m, c):
    return Matrix([[c for _ in range(m)] for _ in range(n)])

def zeros(n, m):
    return constant(n, m, 0)

def ones(n, m):
    return constant(n, m, 1)

def eye(n):
    M = zeros(n, n)
    for i in range(n):
        M[i, i] = 1
    return M

In [ ]:
"""
# Original Attempt that was refined by ChatGPT. This code did not run.

class Matrix:
    def __init__(self, data, m=None):
        if isinstance(data, list):
            self._data = data
            self._rows = len(data)
            self._cols = len(data[0])
        else:
            self._rows = data
            self._cols = m
            self._data = [[0]*m]*data

    def shape(self):
        return (self._rows, self._cols)

    def __getitem__(self, key):
        i, j = key
        return self._data[i][j]

    def __setitem__(self, key, value):
        i, j = key
        self._data[i][j] = value

    def to_list(self):
        return self._data

    def transpose(self):
        return Matrix([[self._data[j][i] for j in range(self._rows)] 
                       for i in range(self._cols)])

    def add(self, other):
        return Matrix([[self._data[i][j] + other._data[i][j] 
                        for j in range(self._cols)]
                       for i in range(self._rows)])

    def mat_mult(self, other):
        result = Matrix([[0]*other._cols]*self._rows)
        for i in range(self._rows):
            for j in range(other._cols):
                for k in range(self._cols):
                    result[i][j] += self._data[i][k] * other._data[k][j]
        return result

    def __repr__(self):
        return str(self._data)
""""

In [6]:
# Forming sample matrices

A = Matrix([[1, 0], [3, 4]])
B = Matrix([[3, 4], [1, 0]])
C = Matrix([[2, 2], [2, 2]])
I = eye(2)

print("A:\n", A)
print("B:\n", B)
print("C:\n", C)
print("I:\n", I)



A:
 [1.0, 0.0]
[3.0, 4.0]
B:
 [3.0, 4.0]
[1.0, 0.0]
C:
 [2.0, 2.0]
[2.0, 2.0]
I:
 [1.0, 0.0]
[0.0, 1.0]


In [9]:
# Test code

print("Shape of A:", A.shape())
print("Transpose of A:\n", A.transpose())
print("Row 0 of A:\n", A.row(0))
print("Column 1 of A:\n", A.column(1))
print("Block of A (0:2, 0:1):\n", A.block(0, 1, 0, 2))
print("A + B:\n", A + B)
print("A - B:\n", A - B)
print("2A:\n", 2 * A)
print("A * B:\n", A * B)
print("Element-wise A * B:\n", A.element_mult(B))
print("A == A:", A == A)
print("A == B:", A == B)


Shape of A: (2, 2)
Transpose of A:
 [1.0, 3.0]
[0.0, 4.0]
Row 0 of A:
 [1.0, 0.0]
Column 1 of A:
 [0.0]
[4.0]
Block of A (0:2, 0:1):
 [1.0]
[3.0]
A + B:
 [4.0, 4.0]
[4.0, 4.0]
A - B:
 [-2.0, -4.0]
[2.0, 4.0]
2A:
 [2.0, 0.0]
[6.0, 8.0]
A * B:
 [3.0, 4.0]
[13.0, 12.0]
Element-wise A * B:
 [3.0, 0.0]
[3.0, 0.0]
A == A: True
A == B: False


In [8]:
# Matrix properties

left = (A * B) * C                  # Associativity
right = A * (B * C)
print("Associative:", left == right)

left = A * (B + C)                  # Distributivity
right = A * B + A * C
print("Distributive:", left == right)

print("AB == BA:", A * B == B * A)  # Non-commutivity

print("AI == A:", A * I == A)       # Idenity matrix

Associative: True
Distributive: True
AB == BA: False
AI == A: True
